# Watermark Removal — ProPainter

**Input (video):** `INPUT_FOLDER` — alt klasörler dahil  
**Input (JSON):** `OUTPUT_FOLDER/results.json` — watermark_detection çıktısı  
**Output:** `OUTPUT_FOLDER/` — orijinal subfolder yapısı korunarak

JSON'daki `videos` key'leri göreli yoldur (`subfolder/video.mp4`).
Her video aynı göreli yola output'a yazılır, eksik klasörler otomatik oluşturulur.


## 1 — Config + Kurulum

In [ ]:
import os, json, time, shutil, subprocess, gc
import cv2
import numpy as np
import torch
from pathlib import Path
from IPython.display import Video, display, HTML

# --- Drive ---
if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

# --- Yollar ---
INPUT_FOLDER  = Path("/content/drive/MyDrive/Watermark-Input")
OUTPUT_FOLDER = Path("/content/drive/MyDrive/Watermark-Output")
JSON_PATH     = OUTPUT_FOLDER / "results.json"
OUTPUT_DIR    = OUTPUT_FOLDER
WORK_DIR      = Path("/content/wm_work")
PP_ROOT       = Path("/content/ProPainter")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

# --- VRAM ayari ---
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# --- Parametreler ---
MASK_DILATION = 5       # manuel mask genisletme (piksel)
PP_NEIGHBOR   = 10
PP_REF_STRIDE = 10
PP_SUBVIDEO   = 80      # OOM onlemek icin 80'den dusuruldu
PP_RAFT_ITER  = 15      # statik watermark icin 20 overkill
PP_MAX_SIZE   = 960     # uzun kenar bu pikseli gecerse kucultulur
OOM_FALLBACK  = [720, 540]  # OOM olursa kademeli kucultme

# --- ProPainter Kurulum ---
if not PP_ROOT.exists():
    !git clone https://github.com/sczhou/ProPainter.git {PP_ROOT}

!pip install -q -r {PP_ROOT}/requirements.txt
!pip install -q timm einops av

WEIGHTS_DIR = PP_ROOT / "weights"
WEIGHTS_DIR.mkdir(exist_ok=True)
WEIGHT_URLS = {
    "ProPainter.pth": "https://huggingface.co/camenduru/ProPainter/resolve/main/ProPainter.pth",
    "recurrent_flow_completion.pth": "https://huggingface.co/camenduru/ProPainter/resolve/main/recurrent_flow_completion.pth",
    "raft-things.pth": "https://huggingface.co/camenduru/ProPainter/resolve/main/raft-things.pth",
    "i3d_rgb_imagenet.pt": "https://huggingface.co/camenduru/ProPainter/resolve/main/i3d_rgb_imagenet.pt",
}
for fname, url in WEIGHT_URLS.items():
    fpath = WEIGHTS_DIR / fname
    if not fpath.exists():
        print(f"Indiriliyor: {fname}")
        !wget -q -L -O {fpath} {url}

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK'}")
print(f"JSON  : {JSON_PATH}")
print(f"Output: {OUTPUT_DIR}")


## 2 — JSON Oku

In [ ]:
with open(JSON_PATH) as f:
    data = json.load(f)

videos     = data["videos"]
wm_list    = {k: v for k, v in videos.items() if v["status"] == "WATERMARKED"}
clean_list = {k: v for k, v in videos.items() if v["status"] == "CLEAN"}

print(f"Toplam: {len(videos)} | WATERMARKED: {len(wm_list)} | CLEAN: {len(clean_list)}")

## 3 — Fonksiyonlar

In [ ]:
# ProPainter'in destekledigi video formatlari
PP_SUPPORTED_EXTS = {".mp4", ".avi", ".mov"}


def convert_to_mp4(video_path, tmp_path):
    """Desteklenmeyen formati (webm vb.) gecici sessiz mp4'e cevirir."""
    result = subprocess.run([
        "ffmpeg", "-y",
        "-i", str(video_path),
        "-an",
        "-c:v", "libx264", "-preset", "fast", "-crf", "18",
        str(tmp_path),
        "-loglevel", "error"
    ], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg convert basarisiz: {result.stderr[-300:]}")


def convert_to_mp4_silent(video_path, tmp_path):
    """Desteklenmeyen formati sessiz mp4'e cevirir (ses yok)."""
    result = subprocess.run([
        "ffmpeg", "-y",
        "-i", str(video_path),
        "-an",
        "-c:v", "libx264", "-preset", "fast", "-crf", "18",
        str(tmp_path),
        "-loglevel", "error"
    ], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg convert basarisiz: {result.stderr[-300:]}")


def create_mask(regions, width, height):
    """Tum watermark bolgelerini tek mask PNG'ye cizer."""
    mask = np.zeros((height, width), dtype=np.uint8)
    for r in regions:
        x1, y1, x2, y2 = r["bbox"]
        mask[y1:y2, x1:x2] = 255
    if MASK_DILATION > 0:
        kernel = np.ones((MASK_DILATION, MASK_DILATION), np.uint8)
        mask = cv2.dilate(mask, kernel, iterations=1)
    return mask


def calc_resize_ratio(w, h, max_size=None):
    """Uzun kenar max_size'i geciyorsa kucultme orani hesapla."""
    if max_size is None:
        max_size = PP_MAX_SIZE
    long_side = max(w, h)
    if long_side <= max_size:
        return 1.0
    return round(max_size / long_side, 2)


def run_propainter(video_path, mask_path, output_dir, resize_ratio=1.0):
    """ProPainter inference: video + mask -> inpaint_out.mp4"""
    cmd = [
        "python", str(PP_ROOT / "inference_propainter.py"),
        "--video", str(video_path),
        "--mask", str(mask_path),
        "--output", str(output_dir),
        "--mask_dilation", "0",
        "--resize_ratio", str(resize_ratio),
        "--neighbor_length", str(PP_NEIGHBOR),
        "--ref_stride", str(PP_REF_STRIDE),
        "--subvideo_length", str(PP_SUBVIDEO),
        "--raft_iter", str(PP_RAFT_ITER),
        "--fp16",
    ]
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = "0"
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(PP_ROOT), env=env)
    if result.returncode != 0:
        stderr = result.stderr[-1500:]
        if "OutOfMemoryError" in stderr or "out of memory" in stderr.lower():
            raise MemoryError(f"CUDA OOM (ratio={resize_ratio})")
        print(f"  STDOUT: {result.stdout[-1000:]}")
        print(f"  STDERR: {stderr}")
        raise RuntimeError("ProPainter basarisiz")



def remove_watermark(video_path, regions, output_path):
    """Tek video icin: (gerekirse convert) -> mask -> ProPainter -> sessiz cikti.
       OOM olursa kademeli olarak kucultup tekrar dener."""
    t0 = time.time()
    video_path  = Path(video_path)
    pp_out      = WORK_DIR / "pp_output"
    mask_path   = WORK_DIR / "mask.png"
    tmp_mp4     = WORK_DIR / "input_converted.mp4"   # webm vb. icin gecici dosya

    try:
        # --- Format kontrolu: ProPainter sadece mp4/avi/mov destekler ---
        if video_path.suffix.lower() not in PP_SUPPORTED_EXTS:
            print(f"  [{video_path.suffix}] ProPainter desteklemiyor, mp4'e cevriliyor...")
            convert_to_mp4(video_path, tmp_mp4)
            pp_input = tmp_mp4          # ProPainter bu dosyayi isler
        else:
            pp_input = video_path       # desteklenen format, direkt kullan

        # Video boyutlarini oku
        cap = cv2.VideoCapture(str(pp_input))
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.release()

        # Tek mask olustur (tum bolgeler birlesik)
        mask = create_mask(regions, w, h)
        cv2.imwrite(str(mask_path), mask)

        # Denenecek max_size degerleri: PP_MAX_SIZE, sonra fallback'ler
        sizes_to_try = [PP_MAX_SIZE] + OOM_FALLBACK
        last_err = None

        for attempt, max_sz in enumerate(sizes_to_try):
            ratio = calc_resize_ratio(w, h, max_sz)
            eff_w, eff_h = int(w * ratio), int(h * ratio)

            if attempt == 0:
                print(f"  {w}x{h}, {len(regions)} bolge, resize={ratio} ({eff_w}x{eff_h})")
            else:
                print(f"  OOM retry {attempt}: max_size={max_sz}, resize={ratio} ({eff_w}x{eff_h})")

            # Temizlik
            if pp_out.exists():
                shutil.rmtree(pp_out)
            gc.collect()
            torch.cuda.empty_cache()

            try:
                run_propainter(pp_input, mask_path, pp_out, ratio)
                last_err = None
                break  # basarili
            except MemoryError as e:
                last_err = e
                print(f"  OOM: {e}")
                continue

        if last_err is not None:
            raise last_err

        # Sonuc: pp_output/inpaint_out.mp4
        inpaint_mp4 = pp_out / "inpaint_out.mp4"
        if not inpaint_mp4.exists():
            mp4s = list(pp_out.rglob("*.mp4"))
            mp4s = [m for m in mp4s if "mask" not in m.name.lower()]
            if not mp4s:
                raise FileNotFoundError(f"ProPainter ciktisi bulunamadi: {pp_out}")
            inpaint_mp4 = mp4s[0]

        # Sessiz cikti: ProPainter sonucunu direkt tasi
        shutil.copy2(str(inpaint_mp4), str(output_path))

        elapsed = time.time() - t0
        size_mb = Path(output_path).stat().st_size / (1024 * 1024)
        print(f"  Tamamlandi: {elapsed:.1f}sn, {size_mb:.1f}MB")
        return elapsed

    except Exception as e:
        print(f"  HATA: {e}")
        import traceback; traceback.print_exc()
        return -1

    finally:
        if pp_out.exists(): shutil.rmtree(pp_out)
        if mask_path.exists(): mask_path.unlink()
        if tmp_mp4.exists(): tmp_mp4.unlink()   # gecici convert dosyasini temizle
        gc.collect()
        torch.cuda.empty_cache()


print("Fonksiyonlar hazir.")


## 4 — Calistir

In [ ]:
t_start = time.time()
results = []
total = len(wm_list) + len(clean_list)
skipped = 0


def copy_without_audio(src, dst):
    """Video dosyasini ses olmadan kopyalar."""
    result = subprocess.run([
        "ffmpeg", "-y",
        "-i", str(src),
        "-an", "-c:v", "copy",
        str(dst),
        "-loglevel", "error"
    ], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg ses silme basarisiz: {result.stderr[-300:]}")


for i, (name, info) in enumerate(wm_list.items(), 1):
    out_name = str(Path(name).with_suffix(".mp4"))
    out_path = OUTPUT_DIR / out_name

    if out_path.exists() and out_path.stat().st_size > 0:
        print(f"[{i}/{total}] SKIP (zaten var): {out_name}")
        results.append({"video": out_name, "action": "SKIP",
                        "regions": len(info["regions"]), "seconds": 0})
        skipped += 1
        continue

    print(f"\n[{i}/{total}] PP: {name} -> {out_name}")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    elapsed = remove_watermark(info["source_path"], info["regions"], str(out_path))
    results.append({"video": out_name, "action": "PP" if elapsed > 0 else "HATA",
                    "regions": len(info["regions"]), "seconds": round(max(elapsed, 0), 1)})

for i, (name, info) in enumerate(clean_list.items(), len(wm_list) + 1):
    src = Path(info["source_path"])
    out_name = str(Path(name).with_suffix(".mp4"))
    out_path = OUTPUT_DIR / out_name

    if out_path.exists() and out_path.stat().st_size > 0:
        print(f"[{i}/{total}] SKIP (zaten var): {out_name}")
        results.append({"video": out_name, "action": "SKIP", "regions": 0, "seconds": 0})
        skipped += 1
        continue

    out_path.parent.mkdir(parents=True, exist_ok=True)

    if src.suffix.lower() == ".mp4":
        print(f"[{i}/{total}] COPY (sessiz): {name}")
        copy_without_audio(src, out_path)
        results.append({"video": out_name, "action": "COPY", "regions": 0, "seconds": 0})
    else:
        print(f"[{i}/{total}] CONVERT (sessiz): {name} -> {out_name}")
        convert_to_mp4_silent(src, out_path)
        results.append({"video": out_name, "action": "CONVERT", "regions": 0, "seconds": 0})

# --- Ozet ---
print(f"\n{'='*60}")
print(f"{'Video':<45} {'Islem':<8} {'Bolge':>5} {'Sure':>8}")
print("-" * 67)
for r in results:
    sure = f"{r['seconds']}sn" if r['seconds'] > 0 else "-"
    print(f"{r['video']:<45} {r['action']:<8} {r['regions']:>5} {sure:>8}")

n_pp   = sum(1 for r in results if r['action'] == 'PP')
n_err  = sum(1 for r in results if r['action'] == 'HATA')
n_skip = sum(1 for r in results if r['action'] == 'SKIP')
n_conv = sum(1 for r in results if r['action'] == 'CONVERT')
n_copy = sum(1 for r in results if r['action'] == 'COPY')
print(f"\n{n_pp} inpaint | {n_copy} kopya | {n_conv} convert | {n_skip} atlandi | {n_err} hata | Toplam: {time.time()-t_start:.1f}sn")


## 5 — Preview

In [ ]:
gc.collect()
torch.cuda.empty_cache()

VIDEO_EXTS = ('.mp4', '.mov', '.avi', '.mkv')

# rglob ile alt klasorler dahil tum videolari bul
output_videos = sorted(
    f for f in OUTPUT_DIR.rglob("*")
    if f.is_file() and f.suffix.lower() in VIDEO_EXTS
)

print(f"{len(output_videos)} video hazir:")
for i, f in enumerate(output_videos):
    # Goreli yol: OUTPUT_DIR'e gore — wm_list key'leriyle eslesir
    rel = str(f.relative_to(OUTPUT_DIR))
    tag = "PP" if rel in wm_list else "CLEAN"
    print(f"  {i}: {rel} [{tag}]")

def preview(n):
    f = output_videos[n]
    local = str(WORK_DIR / f.name)
    shutil.copy2(str(f), local)
    from IPython.display import Video, display, HTML
    display(HTML(f"<h3>{f.relative_to(OUTPUT_DIR)}</h3>"))
    display(Video(local, embed=True, width=720))
    os.remove(local)

# preview(0)
